In [1]:
import json
from pathlib import Path

import pandas as pd
import yaml
import numpy as np
import structlog

from common import read_day, daterange

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

log = structlog.get_logger()

In [2]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}/id_lookup"

In [3]:
def collect_ids(raw_pings_path: str, id_column: str) -> list[str]:
    ids = []
    for i in days:
        temp = read_day(i, raw_pings_path)
        dids = temp.unique(subset=[id_column], keep="first")[id_column].to_list()
        ids += dids
    return ids


def create_lookup(ids: list[str]) -> dict:
    lookup = {}
    for i, device_id in enumerate(np.unique(ids)):
        lookup[device_id] = i + 1
    return lookup


def create_partitions(lookup: dict[str, int]):
    return {v: k[:2] for k, v in lookup.items()}

In [4]:
try:
    with open(f"{OUTPUT}/id_lookup.json", "r") as fp:
        lookup = json.load(fp)
except FileNotFoundError:
    # runs 30 minutes for Budapest data
    ids = collect_ids(config["dirs"]["raw_source"], id_column=config["id_column"])
    log.info(f"number unique devices: {len(np.unique(ids))}")

    lookup = create_lookup(ids)

    Path(OUTPUT).mkdir(parents=True, exist_ok=True)
    with open(f"{OUTPUT}/id_lookup.json", "w") as fp:
        json.dump(lookup, fp, indent=1)

2026-04-30 14:00:54 [info     ] number unique devices: 1652460


## create partitions from original IDs

In [5]:
partition = create_partitions(lookup)

with open(f"{OUTPUT}/partition.json", "w") as fp:
    json.dump(partition, fp, indent=1)

### descriptives

In [11]:
partition_distribution = (
    pd.DataFrame({"partition": partition.values(), "device_id": partition.keys()})
    .groupby("partition")["device_id"]
    .count()
    .reset_index()
)
partition_distribution["device_id"].describe()

count    256.000000
mean     377.832031
std       19.991349
min      309.000000
25%      364.750000
50%      376.500000
75%      392.000000
max      424.000000
Name: device_id, dtype: float64